### Ablation study on vision models

These experiements are to understand the spatial reasoning capabilities and pitfalls as demonstrated in the "What’s “up” with vision-language models? Investigating their struggle with spatial reasoning" paper but with the current sota models

### Part-1 : testing the succesor models

In [ ]:
!git clone https://github.com/ksk-17/whatsup_vlms.git

In [ ]:
%cd whatsup_vlms
!mkdir -p data outputs

In [ ]:
!pip install -r requirements.txt
!pip install -q -U accelerate sentencepiece
!pip install -q -U git+https://github.com/huggingface/transformers.git

In [ ]:
from transformers import Blip2ForImageTextRetrieval, MetaClip2Model, Siglip2Model
print("all imports OK")

In [ ]:
# smoke test
!python main_aro.py --model-name="siglip2:google/siglip2-so400m-patch14-384" \
    --dataset=Controlled_Images_A --download --batch-size=8 --num_workers=2

In [ ]:
models = [
    "siglip2:google/siglip2-so400m-patch14-384",
    "metaclip2:facebook/metaclip-2-worldwide-huge-quickgelu",
    "blip2-itm:Salesforce/blip2-itm-vit-g",
]
datasets = ["Controlled_Images_A", "Controlled_Images_B", "COCO_QA_one_obj",
            "COCO_QA_two_obj", "VG_QA_one_obj", "VG_QA_two_obj"]

for model in models:
    for dataset in datasets:
        print(f"\n=== {model} on {dataset} ===")
        !python -u main_aro.py --model-name="{model}" --dataset="{dataset}" --download --batch-size=32 --num_workers=4

In [ ]:
import pandas as pd
import glob

files = glob.glob("outputs/*.csv")
all_df = pd.concat([pd.read_csv(f, index_col=0) for f in files], ignore_index=True)

# weighted overall accuracy per (Model, Dataset)
overall = (
    all_df.groupby(["Model", "Dataset"])
    .apply(lambda g: (g["Accuracy"] * g["Count"]).sum() / g["Count"].sum() * 100)
    .reset_index(name="Accuracy")
)

# roll the 6 raw datasets up into the paper's 3 groupings
group_map = {
    "Controlled_Images_A": "What'sUp", "Controlled_Images_B": "What'sUp",
    "COCO_QA_one_obj": "COCO-spatial", "COCO_QA_two_obj": "COCO-spatial",
    "VG_QA_one_obj": "GQA-spatial", "VG_QA_two_obj": "GQA-spatial",
}
overall["Group"] = overall["Dataset"].map(group_map)

summary = overall.groupby(["Model", "Group"])["Accuracy"].mean().unstack()
summary = summary[["What'sUp", "COCO-spatial", "GQA-spatial"]]  # fixed column order
summary["Avg"] = summary.mean(axis=1)
summary = summary.round(1)

summary.loc["XVLM 16M (*paper)"] = [41.9, 65.0, 58.2, 55.0]

print(summary.to_string())

Part - 2

In [ ]:
for model in ["llava-onevision", "qwen2.5-vl"]:
    for dataset in ["Controlled_Images_A", "Controlled_Images_B"]:
        print(f"\n=== {model} on {dataset} ===")
        !python -u vqa_eval.py --model-name="{model}" --dataset="{dataset}"

In [ ]:
import pandas as pd
import glob

files = glob.glob("outputs_vqa/vqa_*.csv")
all_df = pd.concat([pd.read_csv(f) for f in files], ignore_index=True)

# Accuracy per Model x Dataset
summary = (
    all_df.groupby(["Model", "Dataset"])["Correct"]
    .mean()
    .mul(100)
    .round(1)
    .unstack()
)
summary["Avg"] = summary.mean(axis=1).round(1)

print(summary.to_string())